# OISST
This notebookd replaces the GetOISST.Rmd, Get_OISST_data.Rmd, and OISST_means.R files
IT IS WAY FASTER AND SIMPLER (sorry to R)

In [16]:
# import required libraries
import xarray as xr
import matplotlib.pyplot as plt
%matplotlib inline
import geopandas as gpd
import regionmask
import pandas as pn
import numpy as np

In [2]:
# This is the OISST url for the monthly data, note that we are subsetting within the brackets []. 
# In subsequent years you will need to alter the time brackets to reflect more updated data.
url = 'http://psl.noaa.gov/thredds/dodsC/Datasets/noaa.oisst.v2.highres/sst.mon.mean.nc?time[0:1:527],lat[500:1:550],lon[1139:1:1180],sst[0:1:527][500:1:550][1139:1:1180]'

In [34]:
# Opening the data using opendap
SST = xr.open_dataset(url)

In [35]:
# Turning 0-360 longitude to -180 through 180
SST.coords['lon'] = SST.coords['lon']-360

In [8]:
# identify which data are in the nyb polygon
NYB = gpd.read_file('~/Desktop/NYB_Indicators_Calculations/Datasets/Shapefiles/PlanningArea_NYocean_NYSDOS.shp')
NYB = NYB.to_crs('EPSG:4326')

In [9]:
# function for cropping the data to the shape
def crop_nd(data, longitude_name, latitude_name, shape):
    
    # Get the region of interest
    region = regionmask.from_geopandas(shape)
    
    # Create the mask
    mask = region.mask(data[longitude_name].astype('f4'), data[latitude_name].astype('f4'))
    
    # Apply mask to the data
    masked_ds = data.where(mask == region.numbers[0])
    
    return masked_ds 

In [52]:
# Crop the data to the shape
SST_nyb = crop_nd(SST, 'lon', 'lat', NYB)


In [60]:
# seasonal averaging
SST_nyb_seas = SST_nyb.groupby(['time.year','time.season']).mean('time')

In [80]:
# Create the table of data
SST_data = pn.DataFrame(SST_nyb_seas.sst.mean(['lat','lon']), index = SST_nyb_seas.sst.mean(['lat','lon']).year, columns = ['winter','summer','spring','fall'])

In [82]:
# Save the data for plotting
SST_data.to_csv('SST_2025.csv')

In [ ]:

## Now the SST data for the marine heatwaves script


In [4]:
sst = xr.open_dataset('~/Desktop//NYB_Indicators_Calculations/CalculateIndicators/WaterTemperature/data/OISST_hatt-fundy_1981-2025.nc')

In [10]:
sst_nyb_daily = crop_nd(sst, 'lon', 'lat', NYB)

In [11]:
sst_nyb_daily_df = sst_nyb_daily.to_dataframe(['time','lon','lat'])

In [13]:
sst_nyb_daily_df = sst_nyb_daily_df.reset_index()

In [17]:
sst_nyb_daily_df = sst_nyb_daily_df[np.isnan(sst_nyb_daily_df.sst) == False]

In [19]:
sst_nyb_daily_df.reset_index().to_csv('sst_nyb_daily_2025.csv')

In [ ]:
##
